# Ask My Docs RAG Walkthrough (Learning-First)

This notebook uses the reusable backend modules in `src/ask_my_docs` and walks through:

1. Ingest
2. Index
3. Retrieve
4. Rerank
5. Answer
6. Evaluate


In [1]:
from pathlib import Path
import json
import sys

cwd = Path.cwd().resolve()
if (cwd / 'src').exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'src').exists():
    PROJECT_ROOT = cwd.parent
else:
    raise RuntimeError('Could not find project root with src/')

if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print('PROJECT_ROOT =', PROJECT_ROOT)


PROJECT_ROOT = /home/ahmad/AI/Projects/production-rag-ask-my-docs-learning


In [2]:
from ask_my_docs.ingestion import build_chunks_from_docs, chunks_to_dataframe
from ask_my_docs.indexing import build_and_save_indexes
from ask_my_docs.pipeline import AskMyDocsEngine
from ask_my_docs.evaluation import evaluate_engine, load_eval_dataset, load_thresholds, apply_gate
from ask_my_docs.settings import SETTINGS

docs_dir = PROJECT_ROOT / 'data/docs'
index_dir = PROJECT_ROOT / 'artifacts/index'
eval_path = PROJECT_ROOT / 'data/eval/qa.yaml'
thresholds_path = PROJECT_ROOT / 'configs/eval_thresholds.yaml'

index_dir.mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / 'artifacts/eval').mkdir(parents=True, exist_ok=True)


## 1) Ingest


In [3]:
chunks = build_chunks_from_docs(
    docs_dir=docs_dir,
    chunk_size_words=SETTINGS.chunk_size_words,
    chunk_overlap_words=SETTINGS.chunk_overlap_words,
)
chunks_df = chunks_to_dataframe(chunks)
print('num_chunks =', len(chunks))
chunks_df.select(['chunk_id', 'title', 'token_count']).head(10)


2026-06-12 08:36:35.051 | INFO     | ask_my_docs.ingestion:build_chunks_from_docs:75 - Ingested 5 chunks from 5 documents


num_chunks = 5


chunk_id,title,token_count
str,str,i64
"""access_control::chunk-000""","""Access Control""",55
"""billing_disputes::chunk-000""","""Billing Disputes""",74
"""data_retention::chunk-000""","""Data Retention""",52
"""incident_sla::chunk-000""","""Incident Sla""",59
"""onboarding_handover::chunk-000""","""Onboarding Handover""",50


## 2) Index


In [4]:
build_and_save_indexes(
    chunks_df=chunks_df,
    index_dir=index_dir,
    embedding_model=SETTINGS.embedding_model,
    reranker_model=SETTINGS.reranker_model,
)
print('index files:', sorted([p.name for p in index_dir.iterdir()]))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-06-12 08:36:42.939 | INFO     | ask_my_docs.indexing:save_indexes:81 - Saved index artifacts in /home/ahmad/AI/Projects/production-rag-ask-my-docs-learning/artifacts/index


index files: ['bm25.pkl', 'chunks.parquet', 'meta.json', 'vectors.faiss']


## 3) Retrieve + 4) Rerank


In [5]:
engine = AskMyDocsEngine.from_index(
    index_dir=index_dir,
    embedding_model=SETTINGS.embedding_model,
    reranker_model=SETTINGS.reranker_model,
    bm25_weight=SETTINGS.bm25_weight,
    vector_weight=SETTINGS.vector_weight,
    rrf_k=SETTINGS.hybrid_rrf_k,
    candidate_pool_size=SETTINGS.candidate_pool_size,
    default_top_k=SETTINGS.default_top_k,
)

question = 'What is the SLA for invoice disputes?'
trace = engine.retriever.retrieve_with_traces(question, top_k=5)
for c in trace['top_chunks']:
    print({
        'rank': c.rank,
        'chunk_id': c.chunk_id,
        'bm25': c.bm25_score,
        'vector': c.vector_score,
        'hybrid': c.hybrid_score,
        'rerank': c.rerank_score,
    })


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

{'rank': 1, 'chunk_id': 'billing_disputes::chunk-000', 'bm25': 4.294835549443716, 'vector': 0.6590222120285034, 'hybrid': 0.01639344262295082, 'rerank': 6.6769914627075195}
{'rank': 2, 'chunk_id': 'incident_sla::chunk-000', 'bm25': 0.8389952568144017, 'vector': 0.42887383699417114, 'hybrid': 0.016001024065540194, 'rerank': -4.911654472351074}
{'rank': 3, 'chunk_id': 'access_control::chunk-000', 'bm25': 0.5184174978651797, 'vector': 0.15180087089538574, 'hybrid': 0.015749007936507936, 'rerank': -11.322174072265625}
{'rank': 4, 'chunk_id': 'data_retention::chunk-000', 'bm25': 0.3093286261641983, 'vector': 0.1400948166847229, 'hybrid': 0.015504807692307693, 'rerank': -11.326253890991211}
{'rank': 5, 'chunk_id': 'onboarding_handover::chunk-000', 'bm25': 0.8490152791261039, 'vector': 0.13062450289726257, 'hybrid': 0.01575682382133995, 'rerank': -11.402776718139648}


## 5) Answer (Citation-Enforced)


In [6]:
response = engine.ask(question, top_k=5)
print(response.answer)


Answer (citation-enforced):
- Billing Dispute Playbook All invoice disputes must be acknowledged within 1 business day. [1]
- Standard-priority disputes have a target resolution SLA of 3 business days after acknowledgement. [1]
- The formal root cause analysis document is due within 3 business days of service restoration. [2]
- Required evidence for triage includes invoice ID, transaction reference, and the disputed contract line item. [1]

Sources:
[1] Billing Disputes (billing_disputes.md, chunk_id=billing_disputes::chunk-000)
[2] Incident Sla (incident_sla.md, chunk_id=incident_sla::chunk-000)
[3] Access Control (access_control.md, chunk_id=access_control::chunk-000)
[4] Data Retention (data_retention.md, chunk_id=data_retention::chunk-000)
[5] Onboarding Handover (onboarding_handover.md, chunk_id=onboarding_handover::chunk-000)


## 6) Evaluate + Gate


In [7]:
dataset = load_eval_dataset(eval_path)
report = evaluate_engine(engine=engine, dataset=dataset, retrieval_k=5)
metrics = report['metrics']
thresholds = load_thresholds(thresholds_path)
passed, failures = apply_gate(metrics={k: float(v) for k, v in metrics.items()}, thresholds=thresholds)

print('metrics:')
for k, v in metrics.items():
    print(f'- {k}: {float(v):.4f}')

print('\ngate_passed =', passed)
if failures:
    print('failures:', failures)

(PROJECT_ROOT / 'artifacts/eval/report_notebook.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
print('saved report to artifacts/eval/report_notebook.json')


metrics:
- hybrid_recall_at_5: 1.0000
- vector_recall_at_5: 1.0000
- hybrid_vs_vector_recall_delta: 0.0000
- hybrid_mrr_at_5: 1.0000
- citation_coverage: 1.0000
- citation_validity: 1.0000
- keyword_recall: 0.8333

gate_passed = True
saved report to artifacts/eval/report_notebook.json
